# IPO — Identity Preference Optimization
### Direct / Reward-Free Alignment  ·  Colab T4 (16 GB) ready

> **DPO's weak spot:** when preferences are near-deterministic or noisy, its reward is *unbounded* and the KL anchor stops mattering — so it **overfits**.
> **IPO fixes this** by regressing the preference margin toward a **finite target** instead of pushing it through a saturating sigmoid toward infinity.

---

## 1. Deep-Dive Conceptual Roadmap & Dataset Ecosystem

### What it is (precise terminology)
- **IPO (Identity Preference Optimization)** comes from the **ΨPO** framework (*Azar et al., "A General Theoretical Paradigm to Understand Learning from Human Preferences"*). ΨPO generalizes RLHF/DPO through a link function **Ψ** applied to preference probabilities; **IPO is the special case where Ψ = identity**.
- Where DPO wraps the preference in the **Bradley–Terry logistic link** (a sigmoid classifier), IPO **drops the link entirely** and uses a **bounded squared-error regression** on the same policy-vs-reference log-ratio margin.
- The shared quantity is DPO's **implicit-reward margin** `h_θ` — the difference of β-scaled log-ratios between chosen and rejected:
  $$h_\theta(x,y_w,y_l) \;=\; \log\frac{\pi_\theta(y_w|x)}{\pi_{\text{ref}}(y_w|x)} \;-\; \log\frac{\pi_\theta(y_l|x)}{\pi_{\text{ref}}(y_l|x)}$$
- IPO **regresses `h_θ` onto a fixed target `1/(2β)`** rather than maximizing it without bound. That constant target is the mechanism that keeps the KL constraint *alive*.

### One-sentence definition of the mechanics
> **IPO minimizes the squared distance between the policy-vs-reference log-ratio margin and a constant target `1/(2β)`, replacing DPO's saturating sigmoid classification with a bounded regression that keeps the KL-to-reference regularization effective even under noisy or deterministic preferences.**

The loss:
$$\mathcal{L}_{\text{IPO}} = \mathbb{E}_{(x,y_w,y_l)}\Big[\Big(h_\theta(x,y_w,y_l) - \tfrac{1}{2\beta}\Big)^{2}\Big]$$

### The exact engineering problem it solves
- **DPO's Bradley–Terry substitution is leaky.** BT converts *pairwise* preferences into a *pointwise* reward. When a pair is (near-)**deterministic** — `y_w` essentially always beats `y_l`, common in clean or duplicated data — the BT reward that explains it is **+∞**.
- To reach that, DPO keeps **inflating the log-ratio**; the sigmoid **saturates**, its gradient toward `π_ref` **vanishes**, and the **KL penalty is effectively ignored**. The policy drifts arbitrarily far from the reference → **overfitting** and degenerate, out-of-distribution generations.
- **Noisy labels make it worse:** DPO trusts every pair as a hard 0/1 target, so mislabeled pairs get pushed to the same unbounded extreme.
- **IPO's fix:** the target `1/(2β)` is **finite**, and the squared loss is **symmetric and bounded in gradient** — the model *stops* once the margin hits the target instead of chasing infinity. The KL regularization therefore **stays binding**, capping deviation from `π_ref` regardless of how deterministic or noisy the pairs are.

---

### The Human Element — Hugging Face preference datasets (IPO consumes the same pairwise schema)

| HF path | What it is | Why it fits IPO specifically |
|---|---|---|
| **`Anthropic/hh-rlhf`** | **Real human** helpfulness/harmlessness `(chosen, rejected)` pairs from crowdworkers. | Human labels are **noisy and sometimes contradictory** — exactly the regime where DPO overfits and IPO's bounded regression pays off. Same `chosen`/`rejected` prefix-sharing structure. |
| **`HuggingFaceH4/ultrafeedback_binarized`** | The canonical DPO/IPO corpus (Zephyr recipe): GPT-4 scored completions, **highest → `chosen`**, lower → `rejected`, split `train_prefs`. | "**binarized**" = already reduced to the **pairwise** form both DPO and IPO require. Many pairs are **strongly separated** (near-deterministic), which is precisely where DPO's unbounded reward misbehaves and IPO stays controlled. |
| **`argilla/dpo-mix-7k`** | A small (~7k) **curated** mix packaged for direct preference methods. | Tiny + high-signal → a fast **T4** run, identical `(prompt, chosen, rejected)` schema; good for sweeping IPO's `β` quickly. |

**Why the schema is identical to DPO:** IPO changes only the **loss's link function**, not the data. It still needs `(x, y_w, y_l)` triples where two completions **share a prompt** and differ in preference — the margin `h_θ` is meaningless without the pair.

> This notebook trains on **`HuggingFaceH4/ultrafeedback_binarized` / `train_prefs`** with **`loss_type="ipo"`** (Section 3).

---

## 2. Architectural Context Block

### **[Context Block]**

#### The 'Why' — the mathematical reason IPO works
- Both DPO and IPO optimize the same **KL-constrained** objective and share the **implicit reward** `r̂_θ = β·log(π_θ/π_ref)`.
- **DPO** plugs the reward into the **Bradley–Terry / logistic** model and minimizes `−log σ(β·h_θ)`. As a pair becomes deterministic, the optimum drives `β·h_θ → +∞`; `σ` saturates near 1, `∂σ/∂h → 0`, and **nothing pulls the policy back toward `π_ref`** — the KL term goes slack.
- **IPO** removes the sigmoid (**Ψ = identity**) and instead solves a **root-finding / regression** problem: make `h_θ` equal the **finite constant `1/(2β)`**. Because the loss is `(h_θ − 1/(2β))²`, the gradient is `2(h_θ − target)` — it **shrinks to zero at the target** and **reverses sign past it**. The margin can't run away, so the **KL regularization remains binding by construction**.
- **β semantics are inverted vs DPO.** In IPO, β is the **regularization strength** (the paper's τ): the target margin is `1/(2β)`, so **higher β ⇒ smaller target ⇒ stays closer to `π_ref`** (more regularization); lower β ⇒ larger allowed margin.

#### VRAM & Compute Impact
- **Identical to DPO** — IPO is a **one-line loss swap** (`loss_type="ipo"`), not an architectural change.
- **1 trainable policy + 1 reference**, and with **PEFT/LoRA** the reference is the *same base weights with adapters disabled* → **effectively one model in VRAM**.
- **No reward model, no value model, no rollouts.** Offline, deterministic, 2 forward passes/step over `chosen ⧺ rejected` (~2× SFT activations).
- **vs RLHF-PPO:** same massive savings as DPO — PPO's ~4 models + online generation are all gone.

#### Pros & Cons (vs DPO)

**Pros**
- **Overfitting-resistant** — bounded target keeps the KL anchor effective; robust to **noisy / deterministic** preference pairs.
- **No reward blow-up** — margins converge to `1/(2β)` instead of drifting to infinity → fewer degenerate / OOD generations.
- **Drop-in** — same data, same trainer, same memory profile as DPO; just `loss_type="ipo"`.
- **Theoretically grounded** — the identity case of the ΨPO framework, with cleaner regularization guarantees.

**Cons**
- **Can under-fit on clean data** — the finite target may leave preference signal on the table where DPO would (correctly) separate harder.
- **New, inverted `β` to tune** — the `1/(2β)` mapping trips people up; needs its **own sweep**, values don't transfer from DPO.
- **Still off-policy** — learns only from the fixed pairs; no exploration (PPO's advantage).
- **Reference-quality dependent** — a weak `π_ref` still caps final quality.

#### Metrics to watch (TRL emits the same `rewards/*` keys)
- **`rewards/margins`** — for IPO this should **converge toward the target `1/(2β)`**, *not* grow without bound (the tell-tale difference from DPO).
- **`rewards/accuracies`** — fraction of pairs with positive margin; should rise then **plateau** (not race to 1.0).
- **`rewards/chosen`, `rewards/rejected`** — the per-side β·log-ratios.
- **`loss`** — a squared-error curve; smooth decrease toward a floor as margins reach the target.

---

## 3. Production-Grade Implementation (Colab T4, 16 GB)

**QLoRA (4-bit) + TRL `DPOTrainer` with `loss_type="ipo"` + the adapter-disabling reference trick.**

> ⚙️ **One-line difference from DPO:** IPO reuses TRL's `DPOTrainer`; you flip **`loss_type="sigmoid"` → `"ipo"`** in `DPOConfig`. Everything else — model, data, memory flags, reference trick — is unchanged.

> ⚙️ **Reference for free:** `ref_model=None` on a **PEFT** policy makes TRL compute the regularization term by **disabling the LoRA adapters** — the frozen 4-bit base weights *are* `π_ref`.

**Executable pipeline:**

| Step | What | Notes |
|---|---|---|
| 1 | 4-bit `Qwen2.5-0.5B-Instruct` + LoRA + tokenizer | policy **and** (adapters-off) reference |
| 2 | `HuggingFaceH4/ultrafeedback_binarized` → `(prompt, chosen, rejected)` | same schema as DPO |
| 3 | `DPOConfig(loss_type="ipo", beta=…)` | the identity/regression objective |
| 4 | `DPOTrainer.train()` | bounded squared loss |
| 5 | Save adapter · export · inference | ship it |

### Environment Setup

In [ ]:
# Modern, up-to-date stack. On Colab T4, torch+CUDA are preinstalled; we only add the RLHF libs.
%pip install -q -U "transformers>=4.45.0" "trl>=0.12.0" "peft>=0.13.0" accelerate bitsandbytes datasets

In [ ]:
import torch, os
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, set_seed
from peft import LoraConfig
from trl import DPOTrainer, DPOConfig  # IPO is a loss_type of the DPOTrainer

set_seed(42)
# Reduce CUDA fragmentation OOMs on the T4 (holds 2x activations: chosen + rejected).
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"
print("CUDA:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only")

### Step 1 — Quantization, LoRA & Tokenizer

Start from the **Instruct** checkpoint (SFT already done): it is both a coherent **policy** and a coherent **reference** `π_ref`. One 4-bit base + one LoRA config serves both roles (adapters on = policy, adapters off = reference). *(Identical setup to the DPO notebook — IPO differs only in the loss.)*

In [ ]:
# The SFT/Instruct start point = the IPO reference policy (pi_ref).
base_model_id = "Qwen/Qwen2.5-0.5B-Instruct"

# 4-bit NF4: base weights (shared by policy AND reference) live in 4-bit on the T4.
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",              # 4-bit NormalFloat (QLoRA)
    bnb_4bit_use_double_quant=True,         # nested quantization -> extra ~0.4 GB saved
    bnb_4bit_compute_dtype=torch.bfloat16,  # matmuls upcast to bf16
)

tokenizer = AutoTokenizer.from_pretrained(base_model_id)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token  # IPO pads batched chosen/rejected; Qwen needs a pad id

# LoRA = the ONLY trainable tensors. Disabling these adapters reproduces pi_ref exactly.
peft_config = LoraConfig(
    r=16, lora_alpha=32, lora_dropout=0.05, bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj",
                    "gate_proj", "up_proj", "down_proj"],  # attention + MLP projections
)

policy_model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    quantization_config=bnb_config,
    device_map="auto",
    attn_implementation="sdpa",  # mem-efficient attention (T4 has no FlashAttn-2)
)
policy_model.config.use_cache = False  # required with gradient checkpointing
print(policy_model.get_memory_footprint() / 1e9, "GB base (4-bit)")

### Step 2 — Dataset: `HuggingFaceH4/ultrafeedback_binarized`

Same pairwise schema DPO uses. We flatten the conversational `chosen`/`rejected` into DPO/IPO's **standard** triple: a templated **`prompt`** plus the two raw assistant completions.

In [ ]:
# train_prefs = the preference split (test_prefs exists for eval).
raw = load_dataset("HuggingFaceH4/ultrafeedback_binarized", split="train_prefs")
raw = raw.shuffle(seed=42).select(range(800))  # subset so a T4 finishes in a few minutes

def to_pref_triple(ex):
    # ex["chosen"] = [{user}, {assistant}] (single-turn). All but the final assistant
    # message is the SHARED prompt.
    prompt_msgs = ex["chosen"][:-1]
    return {
        # Same chat template the model was instruct-tuned on; ends with the assistant
        # header -> policy & reference see identical prompt formatting.
        "prompt":   tokenizer.apply_chat_template(prompt_msgs, tokenize=False,
                                                  add_generation_prompt=True),
        "chosen":   ex["chosen"][-1]["content"],    # preferred completion (raw text)
        "rejected": ex["rejected"][-1]["content"],  # dispreferred completion (raw text)
    }

# String prompt/chosen/rejected => TRL treats it as STANDARD format (no re-templating).
pref_ds = raw.map(to_pref_triple, remove_columns=raw.column_names)
print(pref_ds)

### Step 3 — `DPOConfig(loss_type="ipo")` & `DPOTrainer`

The **only** change from the DPO notebook is `loss_type="ipo"`. Note the **inverted β**: IPO regresses the margin toward `1/(2β)`, so **higher `beta` ⇒ smaller target ⇒ stronger regularization** (closer to `π_ref`). Sweep it independently of any DPO value.

In [ ]:
dpo_config = DPOConfig(
    output_dir="./ipo_output",
    run_name="ipo-t4",

    # ---- The IPO knobs ----
    loss_type="ipo",   # <<< the ONLY change vs DPO: identity link + squared regression
    beta=0.5,          # IPO regularization (paper's tau): target margin = 1/(2*beta) = 1.0.
                       # HIGHER beta -> smaller target -> hug pi_ref harder. Sweep {0.1, 0.5, 1.0}.

    # ---- Sequence budget (chosen AND rejected tokenized => ~2x activation memory) ----
    max_length=1024,        # cap on prompt + completion

    # ---- T4 16 GB hardening ----
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,                         # effective batch = 8
    gradient_checkpointing=True,
    gradient_checkpointing_kwargs={"use_reentrant": False},
    bf16=True, fp16=False,                                 # bf16 matmuls; no fp16 GradScaler pitfalls
    optim="paged_adamw_8bit",                              # paged 8-bit optimizer => tiny state footprint

    # ---- Optimization schedule ----
    learning_rate=5e-6,          # small LR; preference methods collapse under large LRs
    lr_scheduler_type="cosine",
    warmup_ratio=0.1,
    num_train_epochs=1,
    logging_steps=10,
    save_strategy="no",
    report_to="none",
)

# ref_model=None + peft_config => TRL builds pi_ref by DISABLING the LoRA adapters.
# ONE set of 4-bit base weights is BOTH policy and reference (why this fits a T4).
ipo_trainer = DPOTrainer(
    model=policy_model,
    ref_model=None,
    args=dpo_config,
    train_dataset=pref_ds,
    processing_class=tokenizer,  # TRL >= 0.12 (was `tokenizer=` on older versions)
    peft_config=peft_config,
)

### Step 4 — Train

Unlike DPO, watch **`rewards/margins` converge toward the target `1/(2β)`** (= 1.0 here) rather than growing unbounded — that convergence *is* IPO's regularization working.

In [ ]:
ipo_trainer.train()

# Save the aligned LoRA adapter (a few MB, not GB).
ipo_trainer.save_model("./ipo_aligned_adapter")
tokenizer.save_pretrained("./ipo_aligned_adapter")
print("Saved -> ./ipo_aligned_adapter")

## Export — Download the Aligned Adapter (Optional)

In [ ]:
import shutil, os

folder_to_zip = "./ipo_aligned_adapter"
output_filename = "ipo_aligned_adapter.zip"

shutil.make_archive(output_filename.replace(".zip", ""), "zip", folder_to_zip)
if os.path.exists(output_filename):
    print(f"File: {output_filename}  ({os.path.getsize(output_filename)/1e6:.2f} MB)")
else:
    print("Zip not found — run training + save first.")

### Download to your machine

In [ ]:
from google.colab import files
files.download(output_filename)

### Or back up to Google Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

destination_folder = "/content/drive/MyDrive/colab_models"
if os.path.exists(output_filename):
    os.makedirs(destination_folder, exist_ok=True)
    shutil.copy(output_filename, os.path.join(destination_folder, output_filename))
    print("Backed up to Drive:", destination_folder)
else:
    print("Error: zip not found. Did training + zipping finish?")

---

## Model Usage — Evaluate the IPO-Aligned Policy

Reload the **Instruct base + IPO adapter** and generate with the **same chat template** used in training (format must match, or an Instruct model produces junk regardless of alignment quality).

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import PeftModel

base_model_id = "Qwen/Qwen2.5-0.5B-Instruct"
adapter_path = "./ipo_aligned_adapter"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True, bnb_4bit_compute_dtype=torch.bfloat16,
)

tokenizer = AutoTokenizer.from_pretrained(base_model_id, padding_side="left")
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

base = AutoModelForCausalLM.from_pretrained(
    base_model_id, quantization_config=bnb_config, device_map="auto"
)
model = PeftModel.from_pretrained(base, adapter_path)  # attach the IPO adapter
model.eval()

In [ ]:
# Same chat template the IPO loop used (single user turn).
def generate_response(user_prompt, max_new_tokens=256, temperature=0.7):
    messages = [{"role": "user", "content": user_prompt}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(text, return_tensors="pt").to(model.device)
    with torch.no_grad():
        out = model.generate(
            **inputs, max_new_tokens=max_new_tokens,
            temperature=temperature, do_sample=True, top_p=0.9,
            pad_token_id=tokenizer.pad_token_id,
        )
    # Decode only the newly generated completion (slice off the prompt tokens).
    return tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)

In [ ]:
test_prompts = [
    "Explain why IPO is more robust than DPO on noisy preference data.",
    "My friend has been feeling really down lately. How can I support them?",
]

print("--- IPO-Aligned Responses ---")
for i, p in enumerate(test_prompts, 1):
    print(f"\n[Prompt {i}]: {p}")
    print(f"[Response]: {generate_response(p).strip()}")
    print("-" * 60)